## Install Dependancies

In [ ]:
!pip -q install open3d pyvista tetgen

## Imports and paths to point cloud and output directory

In [ ]:
import os
import numpy as np
import open3d as o3d
import pyvista as pv

INPUT_PCD = "/kaggle/input/datasets/jaspreetchhabra/mvs-frontface-full/fused (2).ply"   # <-- change this
OUTPUT_DIR = "/kaggle/working/meshing_output"

METHOD = "poisson"   # options: "poisson", "delaunay", "both"

In [4]:
# preprocessing
VOXEL_SIZE = None    # set to None to disable downsampling
REMOVE_OUTLIERS = True
NB_NEIGHBORS = 20
STD_RATIO = 2.0

# normal estimation
NORMAL_RADIUS = 0.05
MAX_NN = 30

# poisson params
POISSON_DEPTH = 9
POISSON_SCALE = 1.1
POISSON_LINEAR_FIT = False
POISSON_DENSITY_TRIM = 0.02   # remove lowest 2% density vertices

# delaunay params
DELAUNAY_ALPHA = 0.0          # 0.0 = unconstrained tetrahedralization; try 0.01~0.05 if messy
SURFACE_SMOOTH_ITERS = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# HELPERS
# -----------------------------
def load_point_cloud(path):
    pcd = o3d.io.read_point_cloud(path)
    if pcd.is_empty():
        raise ValueError(f"Failed to load point cloud or point cloud is empty: {path}")
    return pcd

def preprocess_point_cloud(pcd, voxel_size=0.01, remove_outliers=True):
    print(f"[INFO] Original points: {len(pcd.points)}")

    if voxel_size is not None and voxel_size > 0:
        pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
        print(f"[INFO] After voxel downsample: {len(pcd.points)}")

    if remove_outliers:
        pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
        print(f"[INFO] After outlier removal: {len(pcd.points)}")

    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=NORMAL_RADIUS, max_nn=MAX_NN)
    )
    pcd.orient_normals_consistent_tangent_plane(50)

    return pcd

def save_point_cloud(pcd, path):
    ok = o3d.io.write_point_cloud(path, pcd)
    if not ok:
        raise RuntimeError(f"Could not save point cloud to {path}")

def poisson_mesh(pcd, out_dir):
    print("[INFO] Running Poisson reconstruction...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd,
        depth=POISSON_DEPTH,
        scale=POISSON_SCALE,
        linear_fit=POISSON_LINEAR_FIT
    )

    densities = np.asarray(densities)
    density_threshold = np.quantile(densities, POISSON_DENSITY_TRIM)
    vertices_to_remove = densities < density_threshold
    mesh.remove_vertices_by_mask(vertices_to_remove)

    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()
    mesh.remove_non_manifold_edges()
    mesh.compute_vertex_normals()

    out_mesh = os.path.join(out_dir, "mesh_poisson.ply")
    out_obj = os.path.join(out_dir, "mesh_poisson.obj")

    o3d.io.write_triangle_mesh(out_mesh, mesh)
    o3d.io.write_triangle_mesh(out_obj, mesh)

    print(f"[INFO] Poisson mesh saved to:\n  {out_mesh}\n  {out_obj}")
    print(f"[INFO] Poisson mesh vertices: {len(mesh.vertices)}, triangles: {len(mesh.triangles)}")

def delaunay_mesh(pcd, out_dir):
    print("[INFO] Running Delaunay 3D meshing...")

    pts = np.asarray(pcd.points)
    cloud = pv.PolyData(pts)

    # Delaunay tetrahedralization
    vol = cloud.delaunay_3d(alpha=DELAUNAY_ALPHA)

    # Extract outer surface
    surf = vol.extract_surface().triangulate()

    # Optional smoothing
    if SURFACE_SMOOTH_ITERS > 0:
        surf = surf.smooth(n_iter=SURFACE_SMOOTH_ITERS)

    out_vtk = os.path.join(out_dir, "mesh_delaunay.vtk")
    out_ply = os.path.join(out_dir, "mesh_delaunay.ply")
    out_obj = os.path.join(out_dir, "mesh_delaunay.obj")

    surf.save(out_vtk)
    surf.save(out_ply)
    surf.save(out_obj)

    print(f"[INFO] Delaunay surface saved to:\n  {out_vtk}\n  {out_ply}\n  {out_obj}")
    print(f"[INFO] Delaunay mesh points: {surf.n_points}, cells: {surf.n_cells}")

def preview_mesh_o3d(mesh_path):
    mesh = o3d.io.read_triangle_mesh(mesh_path)
    if mesh.is_empty():
        print(f"[WARN] Could not preview mesh: {mesh_path}")
        return
    mesh.compute_vertex_normals()
    print(f"[INFO] Preview mesh loaded: {mesh_path}")
    print(f"[INFO] Vertices: {len(mesh.vertices)}, Triangles: {len(mesh.triangles)}")


pcd = load_point_cloud(INPUT_PCD)
pcd = preprocess_point_cloud(
    pcd,
    voxel_size=VOXEL_SIZE,
    remove_outliers=REMOVE_OUTLIERS
)

clean_pcd_path = os.path.join(OUTPUT_DIR, "preprocessed_point_cloud.ply")
save_point_cloud(pcd, clean_pcd_path)
print(f"[INFO] Preprocessed point cloud saved to: {clean_pcd_path}")

if METHOD in ["poisson", "both"]:
    poisson_mesh(pcd, OUTPUT_DIR)

if METHOD in ["delaunay", "both"]:
    delaunay_mesh(pcd, OUTPUT_DIR)

print("\n[INFO] Done.")
print(f"[INFO] Outputs are in: {OUTPUT_DIR}")
print("[INFO] Kaggle tip: open the files from the right panel or zip the folder for download.")

[INFO] Original points: 6286736
[INFO] After outlier removal: 6132940
[INFO] Preprocessed point cloud saved to: /kaggle/working/meshing_output/preprocessed_point_cloud.ply
[INFO] Running Poisson reconstruction...
[Open3D WARNING] Write OBJ can not include triangle normals.
[INFO] Poisson mesh saved to:
  /kaggle/working/meshing_output/mesh_poisson.ply
  /kaggle/working/meshing_output/mesh_poisson.obj
[INFO] Poisson mesh vertices: 199734, triangles: 398569

[INFO] Done.
[INFO] Outputs are in: /kaggle/working/meshing_output
[INFO] Kaggle tip: open the files from the right panel or zip the folder for download.
